# Obtain Word Embeddings for Token Classifiers

### Table of Contents

[0.](#0) Preprocessing

[1.](#1) Custom with fastText

In [ ]:
import config
import pandas as pd
import numpy as np
import re, os
from pathlib import Path
from gensim.models import FastText
from gensim.utils import tokenize
from gensim import utils
from gensim.test.utils import get_tmpfile

<a id="0"></a>
## 0. Preprocessing

In [ ]:
df_train = pd.read_csv(config.tokc_path+"model_input/token_train.csv", index_col=0)
df_dev = pd.read_csv(config.tokc_path+"model_input/token_validate.csv", index_col=0)
df_test = pd.read_csv(config.tokc_path+"model_input/token_test.csv", index_col=0)
print(df_train.shape, df_dev.shape, df_test.shape)
df_train.head()

Obtain the vocabulary of the annotated data:

In [ ]:
df = pd.concat([df_train, df_dev, df_test])

In [ ]:
unique_tokens = list(set(list(df.token)))
unique_words = [token for token in unique_tokens if token.isalpha()]  # keep tokens with only alphabetic characters
print(len(unique_words), len(unique_tokens))

unique_tokens_lower = [token.lower() if token.isalpha() else token for token in unique_tokens]
unique_tokens_lower = list(set(unique_tokens_lower))
unique_words_lower = [token.lower() for token in unique_words]
unique_words_lower = list(set(unique_words_lower))
print(len(unique_words_lower), len(unique_tokens_lower))

<a id="1"></a>
## 1. Custom with fastText

Train custom word embeddings on metadata descriptions from the University of Edinburgh's Archives catalog (as of October 2020) using fastText.

**References:**
* https://radimrehurek.com/gensim/models/fasttext.html
* https://radimrehurek.com/gensim/auto_examples/tutorials/run_fasttext.html#sphx-glr-auto-examples-tutorials-run-fasttext-py

In [ ]:
dir_path = config.docc_path+"model_input/"

In [ ]:
lowercased = False #True

In [ ]:
class CorpusIterator:
    def __iter__(self):
        file_list = ["train_docs.txt", "validate_docs.txt", "test_docs.txt"]
        for file_name in file_list:
            file_path = dir_path+file_name
            with utils.open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if line != "|\n":
                        if lowercased:
                            yield list(tokenize(line.lower()))
                        else:
                            yield list(tokenize(line))

Define the hyperparameters for the unsupervised training of the fastText model (essentially a word2vec model that uses using character n-grams so subwords can help to assign embeddings to unseen words):

In [ ]:
# Specify training architecture (default = "cbow" for Continuous Bag of Words)
training_architectures = ["cbow", "skipgram"]
training_arch = training_architectures[0]
if training_arch == "cbow":
    sg = 0
elif training_arch == "skipgram":
    sg = 1
# Specify the learning rate (default = 0.025)
alpha = 0.025
# Specify the training objective (default = "ns")
# losses = ["ns", "hs", "softmax"]
# loss = losses[0]
# Specify the number of negative words to sample for 'ns' training objective (default = 5)
negative = 5
# Specify the threshold for downsampling higher-frequency words (default = 0.001)
sample = 0.001
# Specify the word embeddings' dimensions
vector_dimensions = 100 #50 #300
# Specify the context window (default is 5) 
context_window = 5
# Specify the number of epochs (default is 5)
epochs = 5
# Specify the threshold of word occurrences (ignore words that occur less than specified number of times; default = 5)
min_count = 5
# Specify the minimum and maximum length of character ngrams (defaults are 3 and 6)
min_n = 2
max_n = 6  # if 0, no character n-grams (subword vectors) will be used
# Specify the number of buckets for hashing ngrams (default = 2000000) 
bucket = 2000000
# Sort vocabulary by descending frequency (default = 1)
sorted_vocab = 1
# Specify the number of threads to use (default = 12)
# threads = 12

In [ ]:
model = FastText(
    sg=sg, alpha=alpha, negative=negative, sample=sample,
    vector_size=vector_dimensions, window=context_window, 
    epochs=epochs, min_count=min_count, min_n=min_n, 
    max_n=max_n, bucket=bucket, sorted_vocab=sorted_vocab
)

In [ ]:
model.build_vocab(corpus_iterable=CorpusIterator())
total_examples = model.corpus_count

In [ ]:
model.train(corpus_iterable=CorpusIterator(), total_examples=total_examples, epochs=epochs)

In [ ]:
model.wv["recipient"]  # Word embedding for the word 'recipient'

Save the model:

In [ ]:
model_dir = config.fasttext_path
Path(model_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
if lowercased:
    is_lower = "_lowercased"
else:
    is_lower = ""
embedding_file_name = config.fasttext_path+"fasttext_{a}_{d}d{lower}.model".format(a=training_arch, d=vector_dimensions, lower=is_lower)
model.save(embedding_file_name)
print("Saved custom word embeddings file: ", embedding_file_name)

In [ ]:
len(model.wv) 

In [ ]:
type(model.wv.key_to_index)     # Looks good

In [ ]:
"the" in model.wv.key_to_index  # Looks good

In [ ]:
"The" in model.wv.key_to_index  # Looks good